# Identifying Premier League BTTS betting opportunities with the Poisson Distribution

Another method for calculating BTTS odds and probabilities is to use the Poisson distribution to model the number of goals scored in a fixed interval of time, i.e. a 90 minute match. The Poisson distribution is well-suited to the task but does have some drawbacks such as the assumption that goals are completely independent events. In reality goals are not entirely independent events - teams often score two goals in quick succession or concede straight after scoring.

In [1]:
import pandas as pd
from process import get_fixtures, expected_goals_home, expected_goals_away
from utils import get_results
from poisson import poisson_calc
from utils import convert_btts_odds_to_probability

In [2]:
match_data = pd.read_csv("data/processed.csv")
odds_data = pd.read_csv("data/betting_odds.csv")

To calculate BTTS yes we will follow the steps below:

1. Calculate home team attack strength. Home team attack strength = home team average goals at home / league home average goals.
2. Calculate away team attack strength. Away team attack strength = away team average goals when away / league away average goals.
3. Calculate home team defence strength. Home team defence strength = home team average goals conceded at home / league average home goals conceded.
4. Calculate away team defence strength. Away team defence strength = away team average goals conceded when away / league average away goals conceded.
5. Calculate home team expected goals. Home team expected goals, $\lambda_h$ = home team attack strength x away team defence strength x league home average goals.
6. Calculate away team expected goals. Away team expected goals, $\lambda_a$ = away team attack strength x home team defence strength x league away average goals.
7. Calculate the probability of the home team scoring no goals. $$P_h(X=0)=\frac{\lambda_h^0 e^{-\lambda_h}}{0!}$$
8. Calculate the probability of the away team scoring no goals. $$P_a(X=0)=\frac{\lambda_a^0 e^{-\lambda_a}}{0!}$$
9. Calculate the probability of both teams scoring. $$P_{BTTS\ yes} = [1-P_h(X=0)][1-P_a(X=0)]$$

Below we use the model to make some predictions.

In [3]:
gameweek = 14
fixtures_14 = get_fixtures(match_data, gameweek)

home_teams = []
away_teams = []
btts_yes = []
btts_no = []
btts_yes_poisson = []

for home, away in fixtures_14:
    home_teams.append(home)
    away_teams.append(away)
    home_odds_df = odds_data[odds_data["homeTeam"] == home]
    match_row = home_odds_df[home_odds_df["awayTeam"] == away]
    btts_yes.append(match_row["BTTSY"].item())
    btts_no.append(match_row["BTTSN"].item())
    xg_home = expected_goals_home(match_data, home, away, gameweek)
    xg_away = expected_goals_away(match_data, home, away, gameweek)
    btts_yes_poisson.append((1 - poisson_calc(0, xg_home))*(1 - poisson_calc(0, xg_away)))

fixtures_14_df = pd.DataFrame({"Home": home_teams, "Away": away_teams, "btts yes": btts_yes,
                                  "btts no": btts_no})
fixtures_14_df["btts_yes_prob"] = fixtures_14_df.apply(lambda row: convert_btts_odds_to_probability(row["btts yes"]), axis=1)
fixtures_14_df["btts_no_prob"] = fixtures_14_df.apply(lambda row: convert_btts_odds_to_probability(row["btts no"]), axis=1)
fixtures_14_df["btts_yes_prob_poisson"] = btts_yes_poisson
fixtures_14_df = fixtures_14_df.round(3)
fixtures_14_df.head(10)

,Home,Away,btts yes,btts no,btts_yes_prob,btts_no_prob,btts_yes_prob_poisson
0,Ipswich,Crystal Palace,1.72,2.12,0.581,0.472,0.363
1,Leicester,West Ham,1.59,2.35,0.629,0.426,0.644
2,Everton,Wolves,1.76,2.06,0.568,0.485,0.249
3,Manchester City,Nottingham Forest,1.72,2.12,0.581,0.472,0.410
4,Newcastle United,Liverpool,1.59,2.35,0.629,0.426,0.284
5,Southampton,Chelsea,1.60,2.34,0.625,0.427,0.643
6,Arsenal,Manchester United,1.75,2.07,0.571,0.483,0.475
7,Aston Villa,Brentford,1.57,2.41,0.637,0.415,0.373
8,Fulham,Brighton,1.57,2.42,0.637,0.413,0.519
9,Bournemouth,Tottenham,1.41,2.94,0.709,0.340,0.310


In week 14 the model has identified some very attractive BTTS no opportunities: Ipswich vs Crystal Palace, Everton vs Wolves, Newcastle United vs Liverpool.

In [4]:
get_results(match_data, gameweek=14)

,home team,home score,away score,away team,btts
0,Ipswich,0,1,Crystal Palace,False
1,Leicester,3,1,West Ham,True
2,Everton,4,0,Wolves,False
3,Manchester City,3,0,Nottingham Forest,False
4,Newcastle United,3,3,Liverpool,True
5,Southampton,1,5,Chelsea,True
6,Arsenal,2,0,Manchester United,False
7,Aston Villa,3,1,Brentford,True
8,Fulham,3,1,Brighton,True
9,Bournemouth,1,0,Tottenham,False


Viewing the results, we see that two of our three BTTS no bets would have won.

In [5]:
gameweek = 37
fixtures_37 = get_fixtures(match_data, gameweek)

home_teams = []
away_teams = []
btts_yes = []
btts_no = []
btts_yes_poisson = []

for home, away in fixtures_37:
    home_teams.append(home)
    away_teams.append(away)
    home_odds_df = odds_data[odds_data["homeTeam"] == home]
    match_row = home_odds_df[home_odds_df["awayTeam"] == away]
    btts_yes.append(match_row["BTTSY"].item())
    btts_no.append(match_row["BTTSN"].item())
    xg_home = expected_goals_home(match_data, home, away, gameweek)
    xg_away = expected_goals_away(match_data, home, away, gameweek)
    btts_yes_poisson.append((1 - poisson_calc(0, xg_home))*(1 - poisson_calc(0, xg_away)))

fixtures_37_df = pd.DataFrame({"Home": home_teams, "Away": away_teams, "btts yes": btts_yes,
                                  "btts no": btts_no})
fixtures_37_df["btts_yes_prob"] = fixtures_37_df.apply(lambda row: convert_btts_odds_to_probability(row["btts yes"]), axis=1)
fixtures_37_df["btts_no_prob"] = fixtures_37_df.apply(lambda row: convert_btts_odds_to_probability(row["btts no"]), axis=1)
fixtures_37_df["btts_yes_prob_poisson"] = btts_yes_poisson
fixtures_37_df = fixtures_37_df.round(3)
fixtures_37_df.head(10)

,Home,Away,btts yes,btts no,btts_yes_prob,btts_no_prob,btts_yes_prob_poisson
0,Aston Villa,Tottenham,1.68,2.18,0.595,0.459,0.498
1,Chelsea,Manchester United,1.60,2.34,0.625,0.427,0.489
2,Everton,Southampton,1.90,1.90,0.526,0.526,0.436
3,West Ham,Nottingham Forest,1.72,2.12,0.581,0.472,0.533
4,Brentford,Fulham,1.57,2.42,0.637,0.413,0.489
5,Leicester,Ipswich,1.57,2.41,0.637,0.415,0.428
6,Arsenal,Newcastle United,1.59,2.36,0.629,0.424,0.697
7,Brighton,Liverpool,1.43,2.82,0.699,0.355,0.489
8,Crystal Palace,Wolves,1.68,2.18,0.595,0.459,0.151
9,Manchester City,Bournemouth,1.76,2.06,0.568,0.485,0.546


By comparing the bookmaker's probabilities with our model probabilities we have a number of attractive bets: Chelsea vs Manchester United - BTTS no, Brentford vs Fulham - BTTS no, Leicester vs Ipswich - BTTS no, Arsenal vs Newcastle United - BTTS yes, Brighton vs Liverpool - BTTS no, Crystal Palace vs Wolves- BTTS no.

In [7]:
get_results(match_data, gameweek=37)

,home team,home score,away score,away team,btts
0,Aston Villa,2,0,Tottenham,False
1,Chelsea,1,0,Manchester United,False
2,Everton,2,0,Southampton,False
3,West Ham,1,2,Nottingham Forest,True
4,Brentford,2,3,Fulham,True
5,Leicester,2,0,Ipswich,False
6,Arsenal,0,0,Newcastle United,False
7,Brighton,3,2,Liverpool,True
8,Crystal Palace,4,2,Wolves,True
9,Manchester City,3,1,Bournemouth,True


Viewing the results, we see that only two of our bets would have won.